In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time
import torch.nn.functional as F

# Check devices
num_gpus = torch.cuda.device_count()
print(f"Available GPUs: {num_gpus}")
if num_gpus > 0:
    for i in range(num_gpus):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# Prepare dataset and dataloaders
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=256, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

Available GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [2]:
# Base Model for Single GPU and Data Parallelism
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(256 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, 10)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 256 * 4 * 4)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

# Model specifically split for Model Parallelism
class SimpleCNN_MP(nn.Module):
    def __init__(self):
        super(SimpleCNN_MP, self).__init__()
        self.dev0 = 'cuda:0'
        self.dev1 = 'cuda:1'
        
        # Put feature extraction on GPU 0
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2)
        ).to(self.dev0)
        
        # Put classification on GPU 1
        self.classifier = nn.Sequential(
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(512, 10)
        ).to(self.dev1)

    def forward(self, x):
        x = x.to(self.dev0)
        x = self.features(x)
        x = x.view(-1, 256 * 4 * 4)
        x = x.to(self.dev1) # Transfer data to GPU 1
        x = self.classifier(x)
        return x

In [3]:
def train_experiment(model, trainloader, criterion, optimizer, num_epochs=3, is_mp=False):
    model.train()
    start_time = time.time()
    
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in trainloader:
            if is_mp:
                # For Model Parallelism: inputs go to GPU 0, targets go to GPU 1 (where output is)
                inputs = inputs.to('cuda:0')
                labels = labels.to('cuda:1')
            else:
                # For Single / Data Parallelism: everything goes to GPU 0 (DataParallel handles the rest)
                inputs = inputs.to('cuda:0')
                labels = labels.to('cuda:0')

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
        epoch_loss = running_loss / len(trainloader)
        epoch_acc = 100 * correct / total
        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")
            
    total_time = time.time() - start_time
    print(f"Total Training Time: {total_time:.2f} seconds\n")
    return total_time, epoch_loss, epoch_acc

In [4]:
print("--- STARTING SINGLE GPU EXPERIMENT ---")
model_single = SimpleCNN().to('cuda:0')
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_single.parameters(), lr=0.001)

time_single, loss_single, acc_single = train_experiment(
    model_single, trainloader, criterion, optimizer, num_epochs=3, is_mp=False
)

--- STARTING SINGLE GPU EXPERIMENT ---
Epoch [1/3] - Loss: 1.5254, Accuracy: 44.30%
Epoch [2/3] - Loss: 1.0624, Accuracy: 62.16%
Epoch [3/3] - Loss: 0.8530, Accuracy: 69.89%
Total Training Time: 22.87 seconds



In [5]:
print("--- STARTING DATA PARALLELISM EXPERIMENT ---")
model_dp = SimpleCNN()
if torch.cuda.device_count() > 1:
    model_dp = nn.DataParallel(model_dp)
model_dp.to('cuda:0')

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_dp.parameters(), lr=0.001)

time_dp, loss_dp, acc_dp = train_experiment(
    model_dp, trainloader, criterion, optimizer, num_epochs=3, is_mp=False
)

--- STARTING DATA PARALLELISM EXPERIMENT ---
Epoch [1/3] - Loss: 1.4932, Accuracy: 45.65%
Epoch [2/3] - Loss: 1.0376, Accuracy: 62.87%
Epoch [3/3] - Loss: 0.8173, Accuracy: 71.17%
Total Training Time: 25.03 seconds



In [6]:
print("--- STARTING MODEL PARALLELISM EXPERIMENT ---")
model_mp = SimpleCNN_MP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_mp.parameters(), lr=0.001)

time_mp, loss_mp, acc_mp = train_experiment(
    model_mp, trainloader, criterion, optimizer, num_epochs=3, is_mp=True
)

--- STARTING MODEL PARALLELISM EXPERIMENT ---
Epoch [1/3] - Loss: 1.4809, Accuracy: 46.14%
Epoch [2/3] - Loss: 1.0329, Accuracy: 63.08%
Epoch [3/3] - Loss: 0.8217, Accuracy: 71.03%
Total Training Time: 21.73 seconds

